# 04b — D-Fire'ı train'e ekle → `firesmoke_v3`

**Çalışma zamanı: T4 GPU yeterli** (DINOv2 embedding için). Toplam ~30–40 dk; çoğu dosya açma/yazma ve zip.

**Neden:** 05b teşhisi modelin train'deki ~190 sahneyi ezberlediğini gösterdi (fire mAP50 train 0.85, val 0.35). En etkili çare sahne çeşitliliği. D-Fire ~21 bin görüntü, yüzlerce farklı sahne (çoğu yerden çekim). **Sadece train'e** eklenir; val ve test aynen kalır, böylece deneyler v2 ile birebir karşılaştırılabilir.

**Sızıntı riski:** 02'de FASDD ile D-Fire arasında aynı görüntüler bulmuştuk. D-Fire'daki bir görüntü val/test'teki bir kareye benziyorsa train'e girmesi sızıntıdır. Bu yüzden her D-Fire görüntüsü DINOv2 ile val+test'in tamamına karşı ölçülür; 04'teki eşiğin (τ) biraz altına kadar benzer olanlar **atılır** (temkinli taraf).

**Sınıflar:** orijinal D-Fire'da `0=smoke, 1=fire`; bizde `0=fire, 1=smoke`. Eşleme sayıyla değil, dataset'in yaml'ındaki **isimlerle** yapılır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L || echo 'GPU yok — CPU ile devam (embedding ~20 dk)'
!pip -q install -U kagglehub

In [ ]:
import os, re, json, shutil, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as TF

ROOT   = Path('/content/drive/MyDrive/firesmoke')
REP    = ROOT / 'reports'
V2_ZIP = ROOT / 'datasets' / 'firesmoke_v2.zip'
V3_ZIP = ROOT / 'datasets' / 'firesmoke_v3.zip'
DATA2  = Path('/content/firesmoke_v2')
DATA3  = Path('/content/firesmoke_v3')
EMB2   = REP / 'dinov2s_emb.npz'           # 04'te hesaplanan (v1/v2 uid'leri)
EMBD   = REP / 'dinov2s_emb_dfire.npz'     # D-Fire embedding önbelleği

# --- Ayarlar -------------------------------------------------------------------
SLUG     = 'sayedgamal99/smoke-fire-detection-yolo'   # 01'de kullanılan D-Fire Kaggle sürümü
MAX_SIDE = 1280
MARGIN   = 0.02     # atma eşiği = τ − MARGIN (τ'nun biraz altını da at: temkinli)
NEG_FRAC = 0.30     # D-Fire negatiflerinin ne kadarı alınsın (train zaten ~%48 negatif)
SEED     = 0
PRETRAINED = True
# ------------------------------------------------------------------------------

m = re.search(r'τ: ([0-9.]+)', (REP / 'dataset_v2.md').read_text())
TAU = float(m.group(1)) if m else 0.936
THR = TAU - MARGIN
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
print(f'τ = {TAU:.3f} → atma eşiği {THR:.3f} · cihaz {DEV}')

## 1. D-Fire'ı indir ve sınıf isimlerini oku

In [ ]:
import kagglehub
DFIRE = Path(kagglehub.dataset_download(SLUG))
print(DFIRE)
names = None
for y in sorted(DFIRE.rglob('*.yaml')):
    d = yaml.safe_load(y.read_text()) or {}
    if 'names' in d:
        names = d['names']; print('yaml:', y.relative_to(DFIRE), '→ names:', names); break
assert names is not None, 'yaml içinde names bulunamadı'
names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
CMAP = {}
for k, v in names.items():
    v = str(v).lower()
    CMAP[k] = 0 if 'fire' in v else 1 if 'smoke' in v else None
assert sorted(c for c in CMAP.values() if c is not None) == [0, 1], f'eşleme belirsiz: {names}'
print('D-Fire sınıf → bizim sınıf:', {names[k]: ['fire', 'smoke'][c] for k, c in CMAP.items() if c is not None})

## 2. Envanter: görüntü + etiket

In [ ]:
def label_path(img):
    parts = list(img.parts)
    i = len(parts) - 1 - parts[::-1].index('images')
    parts[i] = 'labels'
    return Path(*parts).with_suffix('.txt')

rows, bad = [], 0
imgs = sorted(p for p in DFIRE.rglob('*') if p.suffix.lower() in IMG_EXT and 'images' in p.parts
              and not p.name.startswith('._'))
for p in imgs:
    lp = label_path(p)
    lines = []
    if lp.exists():
        for ln in lp.read_text().split('\n'):
            t = ln.split()
            if not t:
                continue
            try:
                c, x, y, w, h = int(float(t[0])), *map(float, t[1:5])
            except (ValueError, IndexError):
                bad += 1; continue
            c = CMAP.get(c)
            if c is None or w <= 0 or h <= 0:
                bad += 1; continue
            lines.append(f'{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
    rel = p.relative_to(DFIRE)
    uid = 'dfire_' + re.sub(r'[^A-Za-z0-9_-]+', '_', '_'.join(rel.with_suffix('').parts))
    nf = sum(l.startswith('0 ') for l in lines); ns = sum(l.startswith('1 ') for l in lines)
    cat = 'fire+smoke' if nf and ns else 'sadece fire' if nf else 'sadece smoke' if ns else 'negatif'
    osplit = next((s for s in ('train', 'val', 'valid', 'test') if s in rel.parts), '')
    rows.append({'uid': uid, 'src': str(p), 'lab': '\n'.join(lines), 'cat': cat, 'n_fire': nf, 'n_smoke': ns,
                 'orig_split': osplit, 'orig_path': f'dfire/{rel.as_posix()}'})
dfi = pd.DataFrame(rows)
assert dfi.uid.is_unique
print(f'D-Fire: {len(dfi)} görüntü · atlanan bozuk etiket satırı: {bad}')
display(dfi.cat.value_counts())

## 3. DINOv2 vektörleri (04 ile aynı model ve ön işleme)

In [ ]:
t0 = time.time()
if not (DATA2 / 'manifest.csv').exists():
    subprocess.run(['unzip', '-q', '-o', str(V2_ZIP), '-d', str(DATA2.parent)], check=True)
man = pd.read_csv(DATA2 / 'manifest.csv')
z = np.load(EMB2, allow_pickle=False)
pos = {u: i for i, u in enumerate(z['uid'])}
held = man[man.split.isin(['val', 'test'])].reset_index(drop=True)
E_held = np.stack([z['emb'][pos[u]] for u in held.uid]).astype(np.float32)
print(f'v2 val+test: {len(held)} görüntü ({time.time() - t0:.0f} sn)')

MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)
class Imgs(torch.utils.data.Dataset):
    def __init__(self, files): self.files = list(files)
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        with Image.open(self.files[i]) as im:
            im.draft('RGB', (448, 448))
            im = im.convert('RGB').resize((224, 224), Image.BILINEAR)
        x = (np.asarray(im, np.float32) / 255.0 - MEAN) / STD
        return torch.from_numpy(x.transpose(2, 0, 1))

cached = None
if EMBD.exists():
    zz = np.load(EMBD, allow_pickle=False)
    if len(zz['uid']) == len(dfi) and (zz['uid'] == dfi.uid.to_numpy().astype(str)).all():
        cached = zz['emb']
if cached is not None:
    E_df = cached; print('önbellekten:', E_df.shape)
else:
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=PRETRAINED, trust_repo=True).to(DEV).eval()
    if DEV == 'cuda':
        model = model.half()
    dl = torch.utils.data.DataLoader(Imgs(dfi.src), batch_size=128 if DEV == 'cuda' else 32,
                                     num_workers=os.cpu_count() or 2, pin_memory=(DEV == 'cuda'))
    out, t0 = [], time.time()
    with torch.inference_mode():
        for b, x in enumerate(dl, 1):
            x = x.to(DEV, non_blocking=True)
            f = model(x.half() if DEV == 'cuda' else x)
            out.append(TF.normalize(f.float(), dim=1).cpu().numpy().astype(np.float16))
            if b % 40 == 0:
                print(f'{min(b * dl.batch_size, len(dfi))}/{len(dfi)}  {time.time() - t0:.0f} sn')
    E_df = np.concatenate(out)
    np.savez(EMBD, emb=E_df, uid=dfi.uid.to_numpy().astype(str))
    print(f'hesaplandı {E_df.shape} · {time.time() - t0:.0f} sn')

## 4. Val/test'e benzeyenleri at

Her D-Fire görüntüsünün val+test'teki en benzer kareye kosinüs benzerliği. `≥ τ − MARGIN` olanlar atılır. 04'te τ'nun üstünün "aynı sahne" olduğunu gözle doğrulamıştık; burada biraz altını da atıyoruz.

In [ ]:
A = torch.from_numpy(E_df.astype(np.float32)).to(DEV)
B = torch.from_numpy(E_held).to(DEV)
best, arg = [], []
for s in range(0, len(A), 4096):
    v, i = (A[s:s + 4096] @ B.T).max(dim=1)
    best.append(v.cpu().numpy()); arg.append(i.cpu().numpy())
dfi['sim'] = np.concatenate(best); dfi['nn'] = held.uid.to_numpy()[np.concatenate(arg)]
dfi['nn_split'] = held.split.to_numpy()[np.concatenate(arg)]
drop = dfi.sim >= THR
print(f'atılan: {int(drop.sum())} / {len(dfi)}  (τ üstü: {int((dfi.sim >= TAU).sum())})')
display(pd.crosstab(dfi.cat, drop.map({True: 'atıldı', False: 'kaldı'})))
display(dfi[drop].groupby('nn_split').size().rename('en yakın komşunun split\'i'))

def show_pairs(d, title, n=8):
    if not len(d):
        print('çift yok:', title); return
    d = d.sample(min(n, len(d)), random_state=SEED)
    fig, axes = plt.subplots(len(d), 2, figsize=(8, 2.4 * len(d)), squeeze=False)
    h_idx = held.set_index('uid')
    for row, (_, r) in zip(axes, d.iterrows()):
        for ax, path, t in [(row[0], r['src'], f"D-Fire · {r['cat']}"),
                            (row[1], DATA2 / h_idx.file[r['nn']], f"{r['nn_split']} · {r['nn'][:26]} · sim {r['sim']:.3f}")]:
            with Image.open(path) as im:
                im.draft('RGB', (512, 512)); ax.imshow(im.convert('RGB'))
            ax.set_title(t, fontsize=7); ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_pairs(dfi[drop], f'ATILANLAR (sim ≥ {THR:.3f}) — sol D-Fire, sağ val/test')
show_pairs(dfi[(dfi.sim < THR) & (dfi.sim >= THR - 0.05)], f'Eşiğin hemen ALTI, KALANLAR ({THR - 0.05:.3f}–{THR:.3f})')

**Gözle kontrol:** üst grid'de çiftler aynı sahne olmalı (atılması doğru). Alt grid'de farklı sahneler olmalı; alt grid'de de aynı sahne çoksa `MARGIN`'i büyüt (örn. 0.05) ve bu bölümü tekrar çalıştır.

## 5. Seç ve `firesmoke_v3`'ü yaz

- Pozitiflerin hepsi, negatiflerin `NEG_FRAC` kadarı alınır.
- v2 dosyaları kopyalanmaz, **hard link** ile bağlanır (anında, ek disk yok).
- D-Fire görüntüleri `images/train/dfire_*.jpg` olarak yazılır (uzun kenar > `MAX_SIDE` ise küçültülür).

In [ ]:
keep = dfi[~drop]
neg = keep[keep.cat == 'negatif'].sample(frac=NEG_FRAC, random_state=SEED)
sel = pd.concat([keep[keep.cat != 'negatif'], neg]).sort_values('uid').reset_index(drop=True)
print(f'eklenecek D-Fire: {len(sel)}  (pozitif {int((sel.cat != "negatif").sum())}, negatif {len(neg)})')

t0 = time.time()
if DATA3.exists():
    shutil.rmtree(DATA3)
shutil.copytree(DATA2, DATA3, copy_function=os.link,
                ignore=shutil.ignore_patterns('*.npy', '*.cache', '_diag_*'))
WH = []
for k, r in enumerate(sel.itertuples(), 1):
    dst = DATA3 / 'images' / 'train' / f'{r.uid}.jpg'
    with Image.open(r.src) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        if max(im.size) > MAX_SIDE:
            im.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
        im.save(dst, quality=95)
        WH.append(im.size)
    (DATA3 / 'labels' / 'train' / f'{r.uid}.txt').write_text(r.lab + ('\n' if r.lab else ''))
    if k % 3000 == 0:
        print(f'{k}/{len(sel)}  {time.time() - t0:.0f} sn')
sel['W'], sel['H'] = [w for w, _ in WH], [h for _, h in WH]

g0 = int(man.group.max()) + 1
add = pd.DataFrame({'file': 'images/train/' + sel.uid + '.jpg', 'uid': sel.uid, 'source': 'dfire', 'split': 'train',
                    'group': np.arange(g0, g0 + len(sel)), 'orig_split': sel.orig_split, 'cat': sel.cat,
                    'n_fire': sel.n_fire, 'n_smoke': sel.n_smoke, 'W': sel.W, 'H': sel.H,
                    'orig_path': sel.orig_path, 'dino_nn_test': sel.sim})
man3 = pd.concat([man, add], ignore_index=True)
(DATA3 / 'manifest.csv').unlink()       # hard link: üstüne yazarsak v2'ninki de değişir
man3.to_csv(DATA3 / 'manifest.csv', index=False)
for y in DATA3.glob('*.yaml'):          # hard link'i kır, yolu v3'e çevir
    t = y.read_text().replace(str(DATA2), str(DATA3)); y.unlink(); y.write_text(t)
for f in ['README.md']:
    if (DATA3 / f).exists():
        (DATA3 / f).unlink()
for s in ['train', 'val', 'test']:
    n_img = sum(1 for p in (DATA3 / 'images' / s).iterdir() if p.suffix == '.jpg')
    n_lbl = sum(1 for p in (DATA3 / 'labels' / s).iterdir() if p.suffix == '.txt')
    assert n_img == n_lbl == int((man3.split == s).sum()), (s, n_img, n_lbl)
assert set(man3[man3.split != 'train'].uid) == set(man[man.split != 'train'].uid), 'val/test değişmemeli'
print(f'yazıldı ({time.time() - t0:.0f} sn)'); print((DATA3 / 'data.yaml').read_text())

## 6. Rapor ve zip → Drive

In [ ]:
t = man3.pivot_table(index=['split', 'source'], columns='cat', values='uid', aggfunc='count', fill_value=0)
t['toplam'] = t.sum(axis=1)
t = t.reindex(['train', 'val', 'test'], level=0)
lines = ['# Dataset v3 — firesmoke (v2 + D-Fire, sadece train)', '',
         f'- v2 + D-Fire {len(sel)} görüntü (pozitif {int((sel.cat != "negatif").sum())}, negatif {len(neg)}, NEG_FRAC {NEG_FRAC})',
         f'- D-Fire toplam {len(dfi)} · val/test\'e benzediği için atılan {int(drop.sum())} (eşik τ−MARGIN = {THR:.3f}, τ = {TAU:.3f})',
         f'- Sınıf eşlemesi (isimden): {names}',
         '- val ve test v2 ile birebir aynı', '', t.to_markdown()]
rep = '\n'.join(lines)
(DATA3 / 'README.md').write_text(rep)
(REP / 'dataset_v3.md').write_text(rep)
shutil.copy(DATA3 / 'manifest.csv', REP / 'dataset_v3_manifest.csv')

tmp = Path('/content/firesmoke_v3.zip')
if tmp.exists():
    tmp.unlink()
t0 = time.time()
subprocess.run(['zip', '-r', '-0', '-q', str(tmp), DATA3.name], cwd=DATA3.parent, check=True)
shutil.copy(tmp, V3_ZIP); tmp.unlink()
print(f'{V3_ZIP}  {V3_ZIP.stat().st_size / 1e9:.2f} GB  ({time.time() - t0:.0f} sn)\n')
print(rep)

**Bana gönderilecekler:** 1. bölümdeki sınıf eşleme satırı, 4. bölümdeki atılan sayısı ve iki grid için cevabın, en sondaki rapor.

Sonra E05: `05_train.ipynb` ayarlarında `DSET = 'firesmoke_v3'`, `EXP = 'E05_y11s_640_dfire'` (ve E03/E04'ten hangisi iyiyse onun `AUG`'u).